In [ ]:
import os
import rasterio as rio
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.path as mpth
from pathlib import Path

import Functions
import importlib

importlib.reload(Functions)

app_path = Functions.get_input_path() / 'App'
output_folder = app_path / 'Documents'
input_path = app_path / 'Documents' / 'csv'

Collecting the moisture from the excel

In [ ]:
moisture = pd.read_excel(app_path / 'Ground_Campaign_untouched' / 'Flevoland_data' / 'Data_25_fields' / 'Average_Soil_moisture.xlsx',
                        header=1)
moisture.drop(columns=['Unnamed: 0','Unnamed: 1'], inplace=True)
moisture.set_index('Code',inplace=True)
moisture = moisture[moisture.index.notna()]
moisture_date = list(pd.to_datetime(moisture.columns))

moisture = moisture.reset_index().melt(id_vars='Code', var_name='Date', value_name='Moist_situ')
moisture.dropna(subset='Moist_situ', inplace=True)
moisture.set_index(pd.to_datetime(moisture['Date']),inplace=True)
moisture.drop(columns='Date',inplace=True)
display(moisture)


Interpolation of the SSM in situ: in this way we have a daily SSM.

In [ ]:
from scipy.interpolate import pchip_interpolate

lista_dataframe_puliti = []

for c in moisture['Code'].unique():

    df = moisture[moisture['Code'] == c].copy()
    daily_dates = pd.date_range(start=df.index.min(), end=df.index.max(), freq='D')
    df = df.reindex(daily_dates)
    df.reset_index(names='Date', inplace=True)
    df['Code'] = c

    df['Umidita_PCHIP_%'] = df['Moist_situ'].interpolate(method='pchip')

    lista_dataframe_puliti.append(df)

df_generale = pd.concat(lista_dataframe_puliti, ignore_index=True)

df_generale = df_generale[['Date', 'Code', 'Moist_situ', 'Umidita_PCHIP_%']]
df_generale.sort_values(by=['Code', 'Date'], inplace=True, ignore_index=True)

Plot of the SSM for the parcel 2081267: the dots are the in-situ data and the line is the interpolated.

In [ ]:
import matplotlib.pyplot as plt

code = 1841225
d = df_generale[df_generale['Code'] == code]

fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(d['Date'], d['Umidita_PCHIP_%'],
           marker='x', s=45, color='tab:blue', label='PCHIP interpolated')
ax.scatter(d['Date'], d['Moist_situ'],
           s=45, color='tab:green', label='In situ')

ax.set_xlabel('Date', fontsize=14)
ax.set_ylabel('SSM (%)', fontsize=14)
ax.set_title(f'Soil moisture — parcel {code}', fontsize=16, fontweight='bold', pad=10)
ax.tick_params(axis='both', labelsize=12)
ax.legend(fontsize=12)
ax.grid(ls='--', alpha=0.4)

fig.autofmt_xdate(rotation=45)
plt.tight_layout()
#plt.savefig('images/PCHIP.png', dpi=300, bbox_inches='tight')
plt.show()

Collecting the backscatter data from a csv

In [ ]:
sigma_TM = pd.DataFrame(pd.read_csv(input_path / 'TimeSeries_sigma.csv'))
sigma_TM['Date'] = pd.to_datetime(sigma_TM['Date'])
sigma_TM.drop(sigma_TM[sigma_TM['mean'] == 0].index, inplace=True)

sigma_TM = sigma_TM.sort_values(by=['Code', 'Band', 'Date'])

display(sigma_TM)

Merged the SSM (interpolated) with the values of backscatter only on the dates that coincide



In [ ]:
sigma_TM = pd.merge(
    sigma_TM, 
    df_generale, 
    on=['Code', 'Date'], 
    how='right' 
).dropna(subset='mean')

Regression between SSM_PCHIP and VV and VH

In [ ]:
from scipy import stats

results_regression_VV = []

for (codice, banda, name,typ), gruppo in sigma_TM.groupby(['Code', 'Band', 'Name', 'Type']):   

    VV = gruppo['mean'].values
    SSM_ret = gruppo['Umidita_PCHIP_%'].values

    if len(VV) > 2:

        slope, intercept, r_value, p_value, std_err = stats.linregress(VV, SSM_ret)        
        y_pred = slope * VV + intercept
        
        residui = SSM_ret - y_pred
        
        rmsre = np.sqrt(np.mean(residui ** 2))
        
        results_regression_VV.append({
            'Code': codice,
            'Name': name,
            'Band': banda,
            'Type': typ,
            'N_points': len(VV),
            'R_Pearson': r_value,
            'R_2': r_value ** 2,
            'P_value': p_value,
            'Slope': slope,
            'Intercept': intercept,
            'RMSRE': rmsre
        })

df_statistiche_VV = pd.DataFrame(results_regression_VV)
df_statistiche_VV.sort_values('R_2', ascending=False, inplace=True)

Difference between the backscatter t+1 and t

In [ ]:
sigma_TM['diff_mean'] = -sigma_TM.groupby(['Code', 'Band'])['mean'].diff(periods=-1)
sigma_TM['diff_lin'] = 10 ** (sigma_TM['diff_mean']/10)
sigma_TM['T'] = sigma_TM.groupby(['Code', 'Band'])['Date'].diff(periods=-1)

sigma_TM = sigma_TM[sigma_TM['Umidita_PCHIP_%'] < 55]

sigma_TM = sigma_TM.sort_values(by=['Code', 'Band', 'Date']).reset_index(drop=True)

In [ ]:
def alpha_recursion(ssm_arr, ratio_arr,delta_t ,win=5):

    ssm_ret = ssm_arr.copy().astype(float)    

    for b in range(0, len(ssm_ret), win):

        for passo in range(1, win):
    
            i = b + passo
            if i >= len(ssm_ret) :
                break
           
            # si el delta t > 7 el counter se pone a 0 y empieza con un in-situ

            if delta_t[i-1] < -pd.Timedelta(days=7):
                continue
            if not np.isnan(ssm_ret[i-1]) and not np.isnan(ratio_arr[i-1]):
                ssm_ret[i] = ratio_arr[i-1] * ssm_ret[i-1]
            else:
                pass

                
    return ssm_ret

sigma_TM['SSM_retrieved'] = sigma_TM['Umidita_PCHIP_%'].copy()

grouped = sigma_TM.groupby(['Code', 'Band'])

for name, group in grouped:

    ssm_vals = group['Umidita_PCHIP_%'].values
    ratio_vals = group['diff_lin'].values
    delta_t_vals = group['T'].values    
    risultato = alpha_recursion(ssm_vals, ratio_vals, delta_t_vals)
    
    sigma_TM.loc[group.index, 'SSM_retrieved'] = risultato

In [ ]:
sigma_TM = sigma_TM[sigma_TM['SSM_retrieved'] < 55]

sigma_TM_copy = sigma_TM.copy()

sigma_TM = sigma_TM[sigma_TM['SSM_retrieved'] != sigma_TM['Umidita_PCHIP_%']]
sigma_TM = sigma_TM[sigma_TM['T'] > -pd.Timedelta(days=7)]

Linear regression between the SSM (obtained from the regression) and the SSM obtained with the alpha approx

In [ ]:
from scipy import stats

results_regression = []

for (codice, banda, name,typ), gruppo in sigma_TM.groupby(['Code', 'Band', 'Name', 'Type']):   

    moist_situ = gruppo['Umidita_PCHIP_%'].values
    SSM_ret = gruppo['SSM_retrieved'].values

    if len(moist_situ) > 2:

        slope, intercept, r_value, p_value, std_err = stats.linregress(moist_situ, SSM_ret)        
        y_pred = slope * moist_situ + intercept
        
        residui = SSM_ret - y_pred
        
        rmsre = np.sqrt(np.mean(residui ** 2))
        
        results_regression.append({
            'Code': codice,
            'Name': name,
            'Band': banda,
            'Type': typ,
            'N_points': len(moist_situ),
            'R_Pearson': r_value,
            'R_2': r_value ** 2,
            'P_value': p_value,
            'Slope': slope,
            'Intercept': intercept,
            'RMSRE': rmsre
        })

df_statistiche = pd.DataFrame(results_regression)
df_statistiche.sort_values('R_2', ascending=False, inplace=True)

filename = "moisture_reg_xfield.csv"
output_path = output_folder / 'csv' / filename

df_statistiche.to_csv(output_path ,index=False, sep=';', decimal='.')

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# --- colori per coltura ---
colori = {
    'Wheat':     'tab:green',
    'Beets':     'tab:orange',
    'Potatoes':  'tab:blue',
    'Corn':      'tab:red',
    'Grassland': 'tab:purple',
}

# ordina per Pearson crescente (i negativi in basso, i positivi in alto)
d = df_statistiche.sort_values('R_Pearson').copy()

# etichette: codice + polarizzazione
etichette = [f"{int(c)} · {b}" for c, b in zip(d['Code'], d['Band'])]

fig, ax = plt.subplots(figsize=(7.5, 9))

ax.barh(etichette,
        d['R_Pearson'],
        color=[colori[t] for t in d['Type']],
        edgecolor='white', linewidth=0.5)

# linea dello zero: è il confine fra retrieval fisico e retrieval invertito
ax.axvline(0, color='black', lw=1.4, zorder=3)

# soglia indicativa di significatività (opzionale, aiuta la lettura)

ax.set_xlabel('Pearson $R$ (SSM retrieved vs in situ)')
ax.set_xlim(-1, 1)
ax.tick_params(axis='y', labelsize=8)
ax.grid(axis='x', ls='--', alpha=0.35, zorder=0)
ax.set_axisbelow(True)



# --- legenda per coltura ---
handles = [Patch(facecolor=c, label=t) for t, c in colori.items()]
ax.legend(handles=handles, title='Crop type',
          loc='lower right', frameon=True, fontsize=9)

plt.tight_layout()
plt.savefig('pearson_by_parcel.png', dpi=500)
plt.show()

Prova con la pioggia

In [ ]:
df_rainfall = pd.read_csv(r'/home/frank/Desktop/f.chiapperino_local/VALENCIA/SCUOLA/App/Documents//csv/rainfall_total.csv', index_col=0)
df_rainfall.index = pd.to_datetime(df_rainfall.index, format='%Y-%m-%d')

df_weekly = df_rainfall.resample('6D', label='left').sum()
rain_6d = df_rainfall.rolling('6D').sum()
rain_6d.name = 'rain_6d'
rain_6d = rain_6d.reset_index()
rain_6d = rain_6d.rename(columns={'date': 'Date'})

In [ ]:
df = sigma_TM.copy()
df = df.rename(columns={'Umidita_PCHIP_%': 'SSM_insitu',
                        'SSM_retreived':   'SSM_retr'})   # via il % dal nome

df['Date'] = pd.to_datetime(df['Date'])

df = df.merge(rain_6d, on='Date', how='left')
print("NaN pioggia:", df['prec_mm'].isna().sum())   # deve essere 0

df['err']     = df['SSM_retrieved'] - df['SSM_insitu']
df['err_abs'] = df['err'].abs()
display(df)


In [ ]:
display(df[df['Code'] == 1545384.0])

In [ ]:
phen = pd.read_csv(output_folder / 'csv' / 'Averaged_Phenology_Stage.csv')
df_long = phen.melt(
    id_vars="Code",        # la colonna che resta fissa
    var_name="Date",       # nuova colonna con i nomi delle vecchie colonne (le date)
    value_name="Phen"      # nuova colonna con i valori
)
df_long["Date"] = pd.to_datetime(df_long["Date"], format="%Y-%m-%d %H:%M:%S").dt.normalize()

In [ ]:
phen_PCHIP = []

for c in df_long['Code'].unique():

    df = df_long[df_long['Code'] == c].copy()
    df = df.set_index('Date')                    # <-- Date diventa l'indice
    df = df.sort_index()                         # utile per l'interpolazione temporale

    daily_dates = pd.date_range(start=df.index.min(), end=df.index.max(), freq='D')
    df = df.reindex(daily_dates)
    df.reset_index(names='Date', inplace=True)   # ora 'Date' non esiste più come colonna: nessun conflitto
    df['Code'] = c
    n_punti = df['Phen'].notna().sum()

    if n_punti >= 2:
        df['Phen_PCHIP'] = df['Phen'].interpolate(method='pchip')
    else:
        df['Phen_PCHIP'] = df['Phen']
        print(f"Code {c}: solo {n_punti} punto/i osservato/i, PCHIP saltata")

    phen_PCHIP.append(df)

phen_generale = pd.concat(phen_PCHIP, ignore_index=True)

phen_generale = phen_generale[['Date', 'Code', 'Phen', 'Phen_PCHIP']]
phen_generale.sort_values(by=['Code', 'Date'], inplace=True, ignore_index=True)



In [ ]:
display(df)

Here a new regression was calculaded, excluding the outliers (values that are 3*stdv away from the value found with the regression).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy import stats
from matplotlib.backends.backend_pdf import PdfPages



results_regression = []

output_folder.mkdir(parents=True, exist_ok=True)

pdf_filename = output_folder / 'Regression_Moisture_xField.pdf'


with PdfPages(pdf_filename) as pdf:

    for i, (_, row) in enumerate(df_statistiche.iterrows()):

        codice = row['Code']
        banda = row['Band']
        tipo = row['Type']
        
        dati_gruppo = sigma_TM[(sigma_TM['Code'] == codice) & (sigma_TM['Band'] == banda)].sort_values('Date')
        
        x_in_situ = dati_gruppo['Umidita_PCHIP_%'].values
        y_calcolata = dati_gruppo['SSM_retrieved'].values
        date_asse = dati_gruppo['Date'].values

        min_date = dati_gruppo['Date'].min()
        max_date = dati_gruppo['Date'].max()
        rain_filtered = df_rainfall[(df_rainfall.index >= min_date) & (df_rainfall.index <= max_date)].reset_index()
        if len(x_in_situ) > 2:
            
            # OUTLIER SEARCH
            y_pred_grezza = row['Slope'] * x_in_situ + row['Intercept']
            residui = y_calcolata - y_pred_grezza
            
            # THRESHOLD
            soglia = 3 * row['RMSRE']
            
            mask_outlier = np.abs(residui) > soglia
            mask_inlier = np.abs(residui) <= soglia
            n_outliers = np.sum(mask_outlier)
            
            x_puliti = x_in_situ[mask_inlier]
            y_puliti = y_calcolata[mask_inlier]
            
            if len(x_puliti) > 2:
                
                fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))
                
                # REGRSSION W/ OUTLIER
                slope_f, intercept_f, r_f, p_f, std_err = stats.linregress(x_puliti, y_puliti)
                r2_pulito = r_f ** 2

                residui = y_puliti - x_puliti
                
                rmsre_f = np.sqrt(np.mean(residui ** 2))

                results_regression.append({
                    'Code': codice,
                    'Band': banda,
                    'Type': tipo,
                    'N_points': len(x_puliti),
                    'R_Pearson': r_f,
                    'R_2': r2_pulito,
                    'P_value': p_f,
                    'Slope': slope_f,
                    'Intercept': intercept_f,
                    'RMSRE': rmsre_f
                })

                # =================================================================
                # SUBPLOT 1: SCATTER PLOT & REGRESSIONE (ax1)
                # =================================================================
                
                # 1. Disegniamo i punti validi (Blue/Teal)
                
                sns.scatterplot(x=x_puliti, y=y_puliti, 
                                color='#1f77b4', s=60, label='Valid Data', alpha=0.8, ax=ax1)
                
                # 2. Disegniamo gli outlier rimossi (Rosso con 'X')
                if n_outliers > 0:
                    sns.scatterplot(x=x_in_situ[mask_outlier], y=y_calcolata[mask_outlier], 
                                    marker='x', color="#ca4828", s=60, label='Outliers', alpha=0.8, ax=ax1)
                
                # 3. Disegniamo la retta di regressione PULITA
                x_linea = np.linspace(x_puliti.min(), x_puliti.max(), 100)
                y_linea = slope_f * x_linea + intercept_f

                sns.lineplot(x=x_linea, y=y_linea, color="#ca4828", 
                             label=f'$R^2$ = {r2_pulito:.3f}', alpha=0.8, ax=ax1)
                


                ax1.fill_between(x_linea, 
                                 y_linea - std_err, 
                                 y_linea + std_err, 
                                 color="#ca4828", alpha=0.15, label='1 Std. Dev.')

                # Linea di riferimento ideale 1:1
                limiti = [min(x_in_situ.min(), y_calcolata.min()), max(x_in_situ.max(), y_calcolata.max())]
                ax1.plot(limiti, limiti, color='gray', linestyle=':', alpha=0.5, label='Ideal 1:1')
                
                # Formattazione pannello 1
                ax1.set_title(f"{codice} - {banda} - {tipo}",
                              fontsize=11, fontweight='bold')
                ax1.set_xlabel("SSM In Situ (%)", fontsize=10)
                ax1.set_ylabel("SSM Retr (%)", fontsize=10)
                ax1.legend(loc='upper left', fontsize=9)
                ax1.grid(True, linestyle='--', alpha=0.5)

                # =================================================================
                # SUBPLOT 2: ANDAMENTO TEMPORALE (ax2)
                # =================================================================
                
                dati_gruppo = sigma_TM_copy[(sigma_TM_copy['Code'] == codice) & (sigma_TM_copy['Band'] == banda)].sort_values('Date')

                x_in_situ = dati_gruppo['Umidita_PCHIP_%'].values
                y_calcolata = dati_gruppo['SSM_retrieved'].values
                date_asse = dati_gruppo['Date'].values

                mask_uguali = (x_in_situ == y_calcolata)
                
                # Se c'è almeno un punto in cui coincidono, disegniamo le X rosse
                if np.any(mask_uguali):
                    ax2.scatter(date_asse[mask_uguali], y_calcolata[mask_uguali], 
                                marker='x', color="red", s=90, linewidth=2, zorder=6, label='Tie-Point')


                # Grafico linea + marker per l'umidità In Situ (usiamo il verde per distinguerlo bene)
                sns.lineplot(x=date_asse, y=x_in_situ, color='#2ca02c', marker='o', 
                             linewidth=1.8, label='SSM In Situ', ax=ax2)
                
                # Grafico linea + marker per l'umidità Stimata (riprendiamo il blu dei dati validi dello scatter)
                sns.lineplot(x=date_asse, y=y_calcolata, color='#1f77b4', marker='^', linestyle='--',
                             linewidth=1.5, label='SSM Retrieved', ax=ax2)
                
                # Opzionale ma consigliato: marchiamo con una 'X' rossa gli outlier anche sulla linea temporale della stima
                if n_outliers > 0:
                    ax2.scatter(date_asse[mask_outlier], y_calcolata[mask_outlier], 
                                marker='x', color="#ca4828", s=70, zorder=5, label='Outliers Identified')
                ax3 = ax2.twinx()

                # Usa il bar plot nativo di Matplotlib invece di Seaborn
                ax3.bar(rain_filtered['date'], rain_filtered['prec_mm'], color='blue', alpha=0.2, width=0.4) 
                # Nota: puoi regolare 'width' (es. 0.5 o 1.0) in base a quanto vuoi spesse le barre

                ax3.set_ylabel('Rain (mm)', color='blue', fontsize=12)
                ax3.tick_params(axis='y', labelcolor='blue')


                # Formattazione pannello 2
                ax2.set_title("Time Series SSM", fontsize=11, fontweight='bold')
                ax2.set_xlabel("Date", fontsize=10)
                ax2.set_ylabel("SSM (%)", fontsize=10)
                ax2.legend(loc='upper right', fontsize=9)
                ax2.grid(True, linestyle='--', alpha=0.5)
                
                # Ruotiamo le date sull'asse X per non farle sovrapporre
                ax2.tick_params(axis='x', rotation=30)
                
                # Compattiamo il layout globale prima del salvataggio
                plt.tight_layout()
                pdf.savefig()
                plt.close()
